# Results analysis — Police agent (team `amireman`)

Companion notebook to the academic report in `README.md`. It analyses the **real**
measurements committed in this repository; every number is read from a CSV produced by a
run, and nothing is typed in by hand.

**Data provenance — read this first.** Two very different kinds of evidence appear below and
they are never mixed:

| Source | What it is | Files |
|---|---|---|
| **Official counted match** | Real league play against another team over the network, cryptographically audited | `docs/evidence/G020/log_G020_g0*.json` |
| **Local simulation** | Our engine playing itself or scripted opponents offline, for measurement only | `evidence/scenario_matchups.csv`, `docs/research/*.csv` |

A local simulation result is *not* a league result and is never presented as one.


In [ ]:
import csv


def load_csv(path):
    with open(path, encoding="utf-8") as handle:
        return list(csv.DictReader(handle))


matchups = load_csv("evidence/scenario_matchups.csv")
oat = load_csv("docs/research/oat_sensitivity.csv")
horizon = load_csv("docs/research/horizon_interaction.csv")
print(f"{len(matchups)} matchup rows | {len(oat)} OAT rows | {len(horizon)} horizon rows")

## 1. The estimator and its uncertainty

Every rate below is a binomial proportion: $n$ independent scenarios, $k$ successes,
$\hat{p} = k/n$. Reporting $\hat{p}$ alone would be misleading, so each point carries a
**Wilson score interval** rather than the textbook normal interval
$\hat{p} \pm z\sqrt{\hat{p}(1-\hat{p})/n}$.

The reason matters here. The normal interval degenerates exactly where much of our data
lives: at $\hat{p}=1$ it returns zero width, claiming perfect certainty from a finite
sample. The Wilson interval inverts the score test instead,

$$
\text{CI} = \frac{1}{1+\frac{z^2}{n}}\left[\hat{p} + \frac{z^2}{2n}
\;\pm\; z\sqrt{\frac{\hat{p}(1-\hat{p})}{n} + \frac{z^2}{4n^2}}\right],
$$

which stays inside $[0,1]$ and remains informative at the boundaries [Wilson 1927;
Brown, Cai & DasGupta 2001]. With $n=200$ and $k=200$ it gives roughly $[0.981, 1.0]$ —
an honest statement that 200 clean trials still do not prove a rate of exactly 1.


In [ ]:
import math


def wilson(k, n, z=1.96):
    """Wilson score interval for a binomial proportion."""
    if n == 0:
        return (0.0, 0.0)
    p = k / n
    denom = 1 + z * z / n
    centre = (p + z * z / (2 * n)) / denom
    half = z * math.sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / denom
    return (max(0.0, centre - half), min(1.0, centre + half))


for k, n in [(200, 200), (0, 200), (100, 200)]:
    lo, hi = wilson(k, n)
    print(f"k={k:>3}/{n}  p={k / n:.3f}  Wilson 95% = [{lo:.3f}, {hi:.3f}]")

## 2. Strategy comparison — paired scenario benchmark

*Local simulation.* 600 scenarios per matchup, varying grid size, barrier budget and move
limit. `base` is the frozen baseline brain, `cand` the candidate that became the production
brain. Pairing matters: both arms see the **same** scenario set, so the comparison is
within-scenario and removes scenario difficulty as a confounder.

![strategy benchmark](images/chart-strategy-benchmark.png)


In [ ]:
for r in matchups:
    name = r["matchup"]
    print(
        f"{name:<34} rate={float(r['rate']):.4f}  "
        f"CI=[{float(r['ci_lo']):.3f},{float(r['ci_hi']):.3f}]  "
        f"technical={r['technical']} illegal={r['illegal']} p95={r['p95_ms']}ms"
    )

base_cop = float([r for r in matchups if r["matchup"].startswith("A_base_police")][0]["rate"])
cand_cop = float([r for r in matchups if r["matchup"].startswith("B_cand_police")][0]["rate"])
base_thf = float([r for r in matchups if r["matchup"].startswith("A_base_thief")][0]["rate"])
cand_thf = float([r for r in matchups if r["matchup"].startswith("C_cand_thief")][0]["rate"])
print()
print(f"Cop   improvement: {base_cop:.4f} -> {cand_cop:.4f}  (+{cand_cop - base_cop:.4f})")
print(f"Thief improvement: {base_thf:.4f} -> {cand_thf:.4f}  (+{cand_thf - base_thf:.4f})")

**Reading the result.** Both roles improved and the intervals do not overlap, so neither
gain is a sampling artefact:

- Cop win rate **0.2317 → 0.4867** (CIs `[0.197,0.268]` vs `[0.450,0.527]`).
- Thief win rate **0.7683 → 0.9150** (CIs `[0.733,0.803]` vs `[0.892,0.937]`).

Two further columns are worth as much as the rates: `illegal = 0` and `technical = 0`
across all 3,600 scenario-plays. A strategy that won by emitting illegal actions would be
worthless, since the firewall would degrade them in a real match.

Row `D` is the candidate playing *itself*: the Cop wins 0.1667, the Thief 0.8333. Our Thief
dominates our Cop at the agreed 7×7/35-step contract. Section 4 shows that this is not a
weak Cop but a horizon effect.


## 3. OAT parameter sensitivity

*Local simulation.* One-at-a-time sweep: each parameter is moved through a range while all
others stay at the agreed Appendix-F value, 200 seeds per point
(`scripts/param_sweep.py`). OAT is the appropriate first-pass design here because the
parameters are set by a **signed contract** — we cannot vary them jointly in a real match,
so the question is the sensitivity of the operating point, not a full response surface
[Saltelli et al. 2008].

![OAT sensitivity](images/chart-oat-sensitivity.png)


In [ ]:
params = sorted({r["parameter"] for r in oat})
for opp in sorted({r["opponent"] for r in oat}):
    for par in params:
        pts = sorted(
            (r for r in oat if r["opponent"] == opp and r["parameter"] == par),
            key=lambda r: float(r["value"]),
        )
        cells = "  ".join(f"{p['value']}:{float(p['capture_rate']):.2f}" for p in pts)
        print(f"{opp:<10} {par:<16} {cells}")
    print()

**Reading the result.** Against every scripted opponent the Cop captures at a rate of 1.00
at *every* setting of *every* parameter. Barrier budget, move limit and pheromone decay
move the outcome not at all.

That flatness is a finding, not a failed experiment: within the tested envelope the
strategy's advantage over scripted play is not parameter-dependent, so there is no
parameter to tune and no fragile operating point to defend. Reporting it flat is more
useful than hunting for a range that would produce a photogenic curve.

Exactly one cell is not flat — self-play, `grid_size ≥ 11`, where the rate falls to 0.00.
Board size is therefore the **only** parameter this strategy pair is sensitive to, and only
against an opponent as strong as itself. Section 4 isolates the cause.


## 4. Isolating the phase transition: board size × horizon

*Local simulation.* The grid-size collapse has a plausible confound. Enlarging the board
while holding `max_moves = 35` shortens the horizon *relative to the distance the Cop must
cover*: on a $g \times g$ board the worst-case pursuit distance grows as $\Theta(g)$, so a
fixed step budget must eventually stop being enough.

To separate *board size* from *horizon*, both were varied together (60 seeds per point).

![horizon interaction](images/chart-horizon-interaction.png)


In [ ]:
import json
from pathlib import Path

rows = []
for path in sorted(Path("docs/evidence/G020").glob("log_G020_g*.json")):
    summary = json.loads(path.read_text())["summary"]
    rows.append(
        (
            summary["sub_game_number"],
            summary["role"],
            summary["result"],
            summary["steps"],
            summary["audit"]["passed"],
            summary["audit"]["tampered"],
            summary["tokens_total"],
        )
    )

print("sub role    result    steps  audit_ok  tampered  llm_tokens")
for row in sorted(rows):
    print(
        f"{row[0]:>3} {row[1]:<7} {row[2]:<9} {row[3]:>5}  "
        f"{str(row[4]):<8} {str(row[5]):<8} {row[6]:>10}"
    )
print()
print("sub-games won by us      :", sum(1 for r in rows if r[2] in ("survival", "capture")))
print("any tampered log         :", any(r[5] for r in rows))
print("total gameplay LLM tokens:", sum(r[6] for r in rows))

**Reading the result.** The confound is confirmed and the effect is cleanly attributable:

| board | 35 steps | 60 steps | 90 steps |
|---|---:|---:|---:|
| 9×9 | 1.00 | 1.00 | 1.00 |
| 11×11 | **0.00** | 1.00 | 1.00 |
| 13×13 | **0.00** | 1.00 | 1.00 |

Give the Cop 60 steps instead of 35 and capture returns to 1.00 on both larger boards. The
collapse is therefore **not** a strategy defect and not a board-size limit — it is the step
budget binding. At the agreed 7×7/35 contract the operating point sits comfortably inside
the capturing region.

This is the most useful single result in the notebook, and it only became visible because
the flat OAT sweep pointed at the one cell worth interrogating.


## 5. The official counted match (not a simulation)

Everything above is offline measurement. The result that counts is the league match, and it
is read here directly from the committed, cryptographically verified logs.


In [ ]:
from pathlib import Path

rows = []
for p in sorted(Path("docs/evidence/G020").glob("log_G020_g*.json")):
    d = json.loads(p.read_text())
    s = d["summary"]
    rows.append(
        (
            s["sub_game_number"],
            s["role"],
            s["result"],
            s["steps"],
            s["audit"]["passed"],
            s["audit"]["tampered"],
            s["tokens_total"],
        )
    )
print("sub role    result    steps  audit_ok  tampered  llm_tokens")
for r in sorted(rows):
    print(f"{r[0]:>3} {r[1]:<7} {r[2]:<9} {r[3]:>5}  {str(r[4]):<8} {str(r[5]):<8} {r[6]:>10}")
print()
print("sub-games won by us :", sum(1 for r in rows if r[2] in ("survival", "capture")))
print("any tampered log    :", any(r[5] for r in rows))
print("total gameplay LLM tokens:", sum(r[6] for r in rows))

**Result.** `G020` vs `Orcai-MJ` — **90 : 30**, sub-games **6 : 0**, every log verified
untampered, mutual consensus confirmed, and **0** gameplay LLM tokens on either side.

The pattern matches the simulation: our Thief survived the full 35 steps in all three
sub-games it defended, and our Cop captured in 9 steps in all three it pursued — faster
than the self-play benchmark, because the opponent was a deterministic ring runner that
`RingBreakerBrain` models exactly (`docs/PRD_ringbreaker.md`).


## 6. Conclusions

1. Both production brains beat their frozen baselines by margins whose 95% intervals do not
   overlap, with zero illegal or technical outcomes across 3,600 scenario-plays.
2. Within the tested envelope the strategies are **insensitive** to barrier budget, move
   limit and pheromone decay — there is no fragile operating point.
3. The single sensitivity is board size, and it resolves to a **horizon** effect: 60 steps
   restores capture at 11×11 and 13×13.
4. The agreed contract (7×7, 35 steps) sits inside the region where the Cop captures
   reliably and the Thief survives against weaker play.
5. The official G020 result is consistent with all of the above, at zero inference cost.

### Limitations

- Self-play measures our brains against each other; it cannot predict an unseen opponent.
- The scripted opponents are weaker than a real league team, so a 1.00 rate against them
  bounds nothing about league play.
- OAT varies one parameter at a time and so cannot detect interaction effects — precisely
  why section 4 had to vary two jointly once the sweep flagged a suspicious cell.
- One counted match against one opponent is a sample of size one.

### References

1. E. B. Wilson, *Probable inference, the law of succession, and statistical inference*,
   JASA 22(158), 1927.
2. L. D. Brown, T. T. Cai, A. DasGupta, *Interval estimation for a binomial proportion*,
   Statistical Science 16(2), 2001.
3. A. Saltelli et al., *Global Sensitivity Analysis: The Primer*, Wiley, 2008.
4. M. Aigner, M. Fromme, *A game of cops and robbers*, Discrete Applied Mathematics 8(1), 1984.
5. P.-P. Grassé, *La reconstruction du nid et les coordinations inter-individuelles*,
   Insectes Sociaux 6, 1959 — stigmergy.
6. Course rulebook, *Distributed Cops-and-Robbers over a Peer-to-Peer Network*, 2026.
